# Etapa 4 — Métricas comportamentais e consolidação por questão

Esta etapa combina métricas de interação dos estudantes com as métricas psicométricas produzidas pela Etapa 3. O resultado é um dataset consolidado por questão, utilizado nas análises posteriores.

O notebook também executa o projeto `codebench-analytics-full` sobre os datasets da pasta `Extraidos`, mantendo parciais e um controle de retomada para que uma interrupção não obrigue o processamento completo novamente.


In [1]:
import json
import subprocess
import re
from pathlib import Path
import sys
import tarfile # Importa a biblioteca para manipular arquivos .tar.gz
import os # Importa a biblioteca os para manipulação de caminhos

# Caminho raiz do projeto (onde está o pyproject.toml)
PROJECT_DIR = Path("./codebench-analytics-full").resolve()

# O diretório de extraídos está um nível acima do local atual.
EXTRAIDOS_DIR = Path("../Extraidos").resolve()


def extract_tar_gz_files(directory):
    """
    Procura por arquivos .tar.gz em um diretório e os extrai.
    Remove o arquivo .tar.gz após a extração bem-sucedida.
    """
    for item in os.listdir(directory):
        if item.endswith(".tar.gz"):
            tar_path = os.path.join(directory, item)
            print(f"📦 Encontrado arquivo compactado: {tar_path}. Extraindo...")
            try:
                with tarfile.open(tar_path, "r:gz") as tar:
                    # Extrai os arquivos para o mesmo diretório
                    tar.extractall(path=directory)
                print(f"✅ Extração de {item} concluída.")
                # Remove o arquivo .tar.gz após a extração
                os.remove(tar_path)
                print(f"🗑️ Arquivo {item} removido.")
            except Exception as e:
                print(f"❌ Erro ao extrair {tar_path}: {e}")


def extract_year_semester_from_folder(folder_name):
    """
    Extrai ano e semestre do nome da pasta.
    Exemplo: cb_dataset_2018_1_v1.81 -> (2018, 1)
    """
    match = re.search(r'cb_dataset_(\d{4})_(\d+)_v', folder_name)
    if match:
        year = match.group(1)
        semester = match.group(2)
        return year, semester
    return None, None

def discover_datasets():
    """
    Descobre, descompacta e prepara os caminhos dos datasets.
    """
    print(f"Procurando datasets em: {EXTRAIDOS_DIR}")
    
    if not EXTRAIDOS_DIR.exists():
        print(f"❌ Diretório Extraidos não encontrado: {EXTRAIDOS_DIR}")
        sys.exit(1)
    
    dataset_paths = []
    
    # Procurar por pastas que seguem o padrão cb_dataset_*
    for item in EXTRAIDOS_DIR.iterdir():
        if item.is_dir() and item.name.startswith("cb_dataset_"):
            year, semester = extract_year_semester_from_folder(item.name)
            
            if year and semester:
                # Construir o caminho para a pasta interna
                dataset_path = item / f"{year}-{semester}"
                
                if dataset_path.exists():
                    # --- NOVA ETAPA: VERIFICAR E EXTRAIR ARQUIVOS .TAR.GZ ---
                    extract_tar_gz_files(dataset_path)
                    
                    dataset_paths.append(str(dataset_path.resolve()))
                    print(f"✅ Dataset preparado: {item.name} -> {dataset_path}")
                else:
                    print(f"⚠️  Pasta interna não encontrada: {dataset_path}")
            else:
                print(f"⚠️  Não foi possível extrair ano/semestre de: {item.name}")
    
    if not dataset_paths:
        print("❌ Nenhum dataset válido encontrado!")
        sys.exit(1)
    
    # Ordenar os caminhos para processamento consistente
    dataset_paths.sort()
    
    print(f"\n📊 Total de datasets preparados: {len(dataset_paths)}")
    return dataset_paths

dataset_paths = discover_datasets()

# Controle persistente: não apagar outputs anteriores e permitir retomada segura.
CONTROL_PATH = PROJECT_DIR.parent / "output" / "controle_processamento.json"

def carregar_controle():
    if CONTROL_PATH.exists():
        try:
            return json.loads(CONTROL_PATH.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            pass
    return {"versao": 1, "datasets": {}, "etapas": {}}

def salvar_controle(controle):
    CONTROL_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporario = CONTROL_PATH.with_suffix(".tmp")
    temporario.write_text(json.dumps(controle, ensure_ascii=False, indent=2), encoding="utf-8")
    temporario.replace(CONTROL_PATH)

def atualizar_controle_etapa(nome, status, datasets=None, erro=None):
    controle = carregar_controle()
    registro = controle.setdefault("etapas", {}).setdefault(nome, {})
    registro.update({"status": status, "atualizado_em": __import__("datetime").datetime.now().isoformat(timespec="seconds")})
    if datasets is not None:
        registro["datasets"] = list(datasets)
    if erro:
        registro["erro"] = str(erro)
    elif status == "concluido":
        registro.pop("erro", None)
    salvar_controle(controle)

def etapa_concluida(nome):
    return carregar_controle().get("etapas", {}).get(nome, {}).get("status") == "concluido"





Procurando datasets em: D:\Trabalho_Super\QuestionInsight\Extraidos
✅ Dataset preparado: cb_dataset_2016_1_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_1_v1.81\2016-1
✅ Dataset preparado: cb_dataset_2016_2_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_2_v1.81\2016-2
✅ Dataset preparado: cb_dataset_2017_1_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_1_v1.81\2017-1
✅ Dataset preparado: cb_dataset_2017_2_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_2_v1.81\2017-2
✅ Dataset preparado: cb_dataset_2018_1_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2018_1_v1.81\2018-1
✅ Dataset preparado: cb_dataset_2018_2_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2018_2_v1.81\2018-2
✅ Dataset preparado: cb_dataset_2019_1_v1.81 -> D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2019_1_v1.81\2019-1
✅ Dataset preparado: cb_dataset_2019_2_v1.81 -> D:\Trabalho_Super\Quest

## 1. Descoberta dos datasets e controle de execução

A rotina identifica pastas no padrão `cb_dataset_YYYY_S_v...` dentro de `../Extraidos/` e prepara os caminhos internos de cada período. Arquivos compactados encontrados nesses diretórios são extraídos antes da análise.

O arquivo `output/controle_processamento.json` registra o status de `execution`, `action` e `solution`, além dos datasets concluídos e dos arquivos parciais. A execução reutiliza somente parciais que existem fisicamente e marca falhas ou interrupções para permitir retomada segura.


In [2]:
import subprocess
import sys
import os
from pathlib import Path


def executar_com_progresso(command, cwd):
    """Executa a CLI oficial e retransmite stdout/stderr."""
    print("$", " ".join(map(str, command)))
    processo = subprocess.Popen(
        [str(x) for x in command],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    try:
        for linha in processo.stdout:
            print(linha, end="", flush=True)
    finally:
        if processo.stdout is not None:
            processo.stdout.close()
    codigo = processo.wait()
    if codigo != 0:
        raise RuntimeError(f"codebench_analytics terminou com código {codigo}")
    return codigo


def codebench_analytics_bin() -> str:
    """Localiza a CLI instalada no .venv da raiz do projeto.

    O notebook pode ser executado por um kernel Anaconda; por isso não
    usamos sys.executable para descobrir a CLI do projeto.
    """
    import shutil
    nome = "codebench_analytics.exe" if os.name == "nt" else "codebench_analytics"
    raiz_projeto = PROJECT_DIR.parent.parent
    candidatos = [
        raiz_projeto / ".venv" / ("Scripts" if os.name == "nt" else "bin") / nome,
        Path(sys.executable).parent / nome,
    ]
    encontrado = shutil.which(nome)
    if encontrado:
        candidatos.insert(0, Path(encontrado))
    for exe in candidatos:
        if exe.is_file():
            return str(exe)
    raise FileNotFoundError(
        "Executável codebench_analytics não encontrado. Procurados: "
        + "; ".join(str(x) for x in candidatos)
        + ". Instale o pacote local com pip no .venv."
    )


def executar_cli(tipo, caminhos):
    """Usa a interface oficial; não importa módulos internos do pacote."""
    comando = [codebench_analytics_bin(), tipo]
    for caminho in caminhos:
        comando.extend(["-p", str(caminho)])
    comando.extend(["-k", "exam"])
    executar_com_progresso(comando, PROJECT_DIR)


def main():
    executar_cli("execution", dataset_paths)
    executar_cli("action", dataset_paths)
    solution_csv = PROJECT_DIR / "input" / "codigo_solucao.csv"
    if not solution_csv.exists():
        raise FileNotFoundError(f"Arquivo de solução não encontrado: {solution_csv}")
    executar_com_progresso(
        [codebench_analytics_bin(), "solution", "-p", str(solution_csv)],
        PROJECT_DIR,
    )


main()


$ D:\Trabalho_Super\QuestionInsight\.venv\Scripts\codebench_analytics.exe execution -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_1_v1.81\2016-1 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_2_v1.81\2016-2 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_1_v1.81\2017-1 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_2_v1.81\2017-2 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2018_1_v1.81\2018-1 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2018_2_v1.81\2018-2 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2019_1_v1.81\2019-1 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2019_2_v1.81\2019-2 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2021_1_v1.81\2021-1 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2021_2_v1.81\2021-2 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2022_1_v1.81\2022-1 -p D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_

2026-11-06 22:43:35,472 codebench_analytics.extractor.execution_extractor INFO:extracting execution data from 'D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_1_v1.81\2016-1'


2026-11-06 22:44:02,274 codebench_analytics.extractor.execution_extractor INFO:extracting execution data from 'D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_2_v1.81\2016-2'


2026-11-06 22:44:10,144 codebench_analytics.extractor.execution_extractor INFO:extracting execution data from 'D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_1_v1.81\2017-1'


2026-11-06 22:44:43,433 codebench_analytics.extractor.execution_extractor INFO:extracting execution data from 'D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_2_v1.81\2017-2'


2026-11-06 22:44:52,252 codebench_analytics.extractor.execution_extractor INFO:extracting execution data from 'D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2018_1_v1.81\2018-1'


... (44 outputs left out)

2026-11-06 23:46:32,542 root INFO:generating 'actions_by_student.csv' file into 'output/metrics\actions_by_student.csv'


2026-11-06 23:46:32,739 root INFO:generating 'actions_data.csv' file into 'output/metrics\actions_data.csv'


$ D:\Trabalho_Super\QuestionInsight\.venv\Scripts\codebench_analytics.exe solution -p D:\Trabalho_Super\QuestionInsight\Etapa_4\codebench-analytics-full\input\codigo_solucao.csv


2026-11-06 23:46:43,374 root INFO:generating 'code_metrics_professor.csv' file into 'output/data\code_metrics_professor.csv'


## 2. Extração das métricas do CodeBench

Para cada dataset, o notebook extrai dois grupos de eventos do tipo **exam**: executions e ações do CodeMirror. Os dados brutos são processados em diretórios temporários isolados, consolidados somente depois da extração e enviados aos coletores do `codebench-analytics-full`.

A etapa `solution` também processa `codebench-analytics-full/input/codigo_solucao.csv` e gera `code_metrics_professor.csv`. Os resultados intermediários são guardados em diretórios `*_staging`, enquanto as saídas consolidadas ficam em `codebench-analytics-full/output/data/`.


In [3]:
import pandas as pd
import os

# Cria a pasta de saída, caso ainda não exista
os.makedirs('output', exist_ok=True)

# Carregar os dois arquivos CSV
df1 = pd.read_csv(r'codebench-analytics-full/output/metrics/actions_data.csv')  # Primeiro arquivo com os tempos e eventos
df2 = pd.read_csv(r'codebench-analytics-full/output/metrics/executions.csv')  # Segundo arquivo com as interações e resultados

# Unir os DataFrames com base na coluna 'question'
merged_df = pd.merge(df2, df1, on='question', how='left')

# Calcular todas as métricas M1-M13 possíveis
merged_df['taxa_acerto']         = merged_df['num_correct'] / merged_df['num_students_interactions']        # M1
merged_df['num_submissoes']      = merged_df['num_submissions'] / merged_df['num_students_interactions']    # M2
merged_df['taxa_aceitacao']      = merged_df['num_correct'] / merged_df['num_submissions']                  # M3
merged_df['num_testes']          = merged_df['num_tests'] / merged_df['num_students_interactions']          # M4
merged_df['num_consultas']       = (merged_df['num_submissions'] + merged_df['num_tests']) / merged_df['num_students_interactions']  # M5
merged_df['num_erros_lgcs']      = merged_df['num_logic_errors'] / merged_df['num_students_interactions']   # M6
merged_df['num_errors_stx']      = merged_df['num_syntax_errors'] / merged_df['num_students_interactions']  # M7
merged_df['num_erros']           = merged_df['num_errors'] / merged_df['num_students_interactions']         # M8
merged_df['num_eventos']         = merged_df['num_events'] / merged_df['num_students_interactions']         # M9
merged_df['num_eventos_del']     = merged_df['num_deletes'] / merged_df['num_students_interactions']        # M10
merged_df['num_eventos_sem_delecoes'] = merged_df['num_eventos'] - merged_df['num_eventos_del'] # MD13
merged_df['tempo_implementacao'] = merged_df['code_time'] / merged_df['num_correct']                        # M11
merged_df['qtd_alteracoes_codigo'] = merged_df['amount_of_change'] / merged_df['num_students_interactions'] # M12

# Selecionar colunas finais
final_df = merged_df[[
    'question',
    'taxa_acerto',           # M1
    'num_submissoes',        # M2
    'taxa_aceitacao',        # M3
    'num_testes',            # M4
    'num_consultas',         # M5
    'num_erros_lgcs',        # M6
    'num_errors_stx',        # M7
    'num_erros',             # M8
    'num_eventos',           # M9
    'num_eventos_del',       # M10
    'num_eventos_sem_delecoes', # MD13
    'tempo_implementacao',   # M11
    'qtd_alteracoes_codigo', # M12
]]

final_df.to_csv('output/metricas_execucao_por_questao.csv', index=False)
print("Arquivo 'output/metricas_execucao_por_questao.csv' gerado com sucesso!")


Arquivo 'output/metricas_execucao_por_questao.csv' gerado com sucesso!


## 3. Métricas de execução por questão

`actions_data.csv` e `executions.csv` são unidos pela coluna `question`. A partir dessa união, o notebook calcula as métricas comportamentais normalizadas por interações, incluindo taxa de acerto, submissões, testes, consultas, erros lógicos, erros de sintaxe, eventos, exclusões, tempo de implementação e alterações de código.

O resultado é salvo em `output/metricas_execucao_por_questao.csv`. Essas métricas descrevem o comportamento observado nas interações.


In [4]:
import pandas as pd
import os

# Cria a pasta de saida, caso ainda nao exista
os.makedirs('output', exist_ok=True)
os.makedirs('input', exist_ok=True)

# Caminho do arquivo de entrada (externo, nao pertence a nenhuma etapa) e nome do arquivo de saida
input_file = os.path.join('..', 'CSVS_JO', 'unified_solutions.csv')  # gerado por Etapa_1/Etapa_1.ipynb
output_file = 'output/assessments_ids_unicos.csv'

# Carrega o CSV de entrada
df = pd.read_csv(input_file)

# Cria a coluna 'id' a partir da coluna 'assignment'
df['id'] = df['assignment']

# Remove duplicatas da coluna 'id'
df_output = df[['id']].drop_duplicates()

# Salva o novo DataFrame em um arquivo CSV
df_output.to_csv(output_file, index=False)

print(f"Arquivo '{output_file}' criado com sucesso!")


Arquivo 'output/assessments_ids_unicos.csv' criado com sucesso!


## 4. Identificadores únicos de assessments

O arquivo `../CSVS_JO/unified_solutions.csv`, produzido anteriormente, é usado para criar `output/assessments_ids_unicos.csv`. A rotina copia `assignment` para a coluna `id` e remove duplicatas.

Esse arquivo é um catálogo auxiliar de assessments. Ele não altera o índice filtrado da Etapa 2 nem substitui as referências do `referencias_processamento.json`.


In [5]:
import csv
import os

os.makedirs('output', exist_ok=True)

def ler_csv(nome_arquivo):
    with open(nome_arquivo, mode='r', newline='') as arquivo_csv:
        leitor = csv.DictReader(arquivo_csv)
        return [linha for linha in leitor]

dados_question = ler_csv(r'output/metricas_execucao_por_questao.csv')
dados_respostas = ler_csv(r'../Etapa_3/output/questoes_ordenadas.csv')

dados_respostas = ler_csv(r'../Etapa_3/output/questoes_ordenadas.csv')
# Índice por ID: substitui a busca aninhada O(N*M) por buscas O(1),
# preservando a mesma regra de correspondência por questão.
respostas_por_id = {int(resposta['id']): resposta for resposta in dados_respostas}
resultados = []
for question in dados_question:
    resposta = respostas_por_id.get(int(question['question']))
    if resposta is None:
        continue
    novo_registro = {
        "question":              question['question'],
        "taxa_acerto":           question['taxa_acerto'],
        "num_submissoes":        question['num_submissoes'],
        "taxa_aceitacao":        question['taxa_aceitacao'],
        "num_testes":            question['num_testes'],
        "num_consultas":         question['num_consultas'],
        "num_erros_lgcs":        question['num_erros_lgcs'],
        "num_errors_stx":        question['num_errors_stx'],
        "num_erros":             question['num_erros'],
        "num_eventos":           question['num_eventos'],
        "num_eventos_del":       question['num_eventos_del'],
        "tempo_implementacao":   question['tempo_implementacao'],
        "qtd_alteracoes_codigo": question['qtd_alteracoes_codigo'],
        "dificuldade":           resposta['dificuldade'],
        "respostas":             resposta['respostas'],
        "usuarios_respondidos":  resposta['usuarios_respondidos']
    }
    resultados.append(novo_registro)
with open('output/metricas_questoes_com_dificuldade.csv', mode='w', newline='') as arquivo_csv:
    campos = resultados[0].keys()
    escritor_csv = csv.DictWriter(arquivo_csv, fieldnames=campos)
    escritor_csv.writeheader()
    escritor_csv.writerows(resultados)

print("Dados combinados e salvos em output/metricas_questoes_com_dificuldade.csv")

Dados combinados e salvos em output/metricas_questoes_com_dificuldade.csv


## 5. Inclusão das métricas psicométricas da Etapa 3

`metricas_execucao_por_questao.csv` é associado a `../Etapa_3/output/questoes_ordenadas.csv` pelo ID da questão. A junção adiciona dificuldade, quantidade de respostas e usuários respondentes às métricas comportamentais.

Questões sem correspondência na saída da Etapa 3 são ignoradas. O resultado é salvo em `output/metricas_questoes_com_dificuldade.csv` e ordenado pelo campo `question`.


In [6]:
import pandas as pd

def ordenar_question(arquivo_question):
    # Ler o arquivo CSV
    question_df = pd.read_csv(arquivo_question)

    # Ordenar o DataFrame pela coluna 'question'
    question_df_sorted = question_df.sort_values(by='question', ascending=True)

    # Sobrescrever o arquivo original com o DataFrame ordenado
    question_df_sorted.to_csv(arquivo_question, index=False)

    print(f"Arquivo '{arquivo_question}' foi ordenado e salvo com sucesso.")

# Exemplo de uso
arquivo_question = 'output/metricas_questoes_com_dificuldade.csv'
ordenar_question(arquivo_question)


Arquivo 'output/metricas_questoes_com_dificuldade.csv' foi ordenado e salvo com sucesso.


## 6. Dataset consolidado final

A saída com dificuldade é combinada com `codebench-analytics-full/output/data/code_metrics_professor.csv` usando o identificador da questão. A junção é interna, portanto somente questões presentes nas duas fontes permanecem no resultado.

O arquivo final `output/dataset_questoes_consolidado.csv` reúne as métricas de execução, as métricas psicométricas e as métricas derivadas da solução do professor. Ao final, a coluna temporária `question_id` volta a se chamar `question` para manter o contrato usado pelas análises posteriores.


In [7]:
import pandas as pd

def comparar_e_gerar_csv(arquivo_question, arquivo_metrics, arquivo_saida):
    # Ler os arquivos CSV
    question_df = pd.read_csv(arquivo_question)
    metrics_df = pd.read_csv(arquivo_metrics)

    # Renomear a coluna 'question' para 'question_id' em question_df para facilitar a junção
    question_df.rename(columns={'question': 'question_id'}, inplace=True)

    # Fazer a junção dos DataFrames com base na coluna 'question_id'
    merged_df = pd.merge(metrics_df, question_df, on='question_id', how='inner')

    # Ordenar o DataFrame resultante com base na ordem de 'question_id' do question_df original
    merged_df = merged_df.sort_values(by='question_id')

    # Selecionar as colunas que você deseja manter
    colunas_a_manter = metrics_df.columns.tolist()  # Manter todas as colunas do metrics_df

    # Salvar o DataFrame resultante em um novo arquivo CSV
    merged_df.to_csv(arquivo_saida, columns=colunas_a_manter, index=False)

# Exemplo de uso com uma string "raw"
comparar_e_gerar_csv(
    'output/metricas_questoes_com_dificuldade.csv', 
    r'codebench-analytics-full/output/data/code_metrics_professor.csv', #Etapa_4/codebench-analytics-full/output
    'output/dataset_questoes_consolidado.csv'
)

print("Arquivo gerado: output/dataset_questoes_consolidado.csv")



# Ler o arquivo CSV
df = pd.read_csv('output/dataset_questoes_consolidado.csv')

# Renomear a coluna 'question_id' para 'question'
df.rename(columns={'question_id': 'question'}, inplace=True)

# Salvar o DataFrame com o novo nome da coluna
df.to_csv('output/dataset_questoes_consolidado.csv', index=False)

print("Nome da coluna alterado com sucesso no arquivo 'output/dataset_questoes_consolidado.csv'.")


Arquivo gerado: output/dataset_questoes_consolidado.csv


Nome da coluna alterado com sucesso no arquivo 'output/dataset_questoes_consolidado.csv'.
